
1. What is Lakeflow?
Lakeflow is Databricks' data engineering solution, with three main capabilities:
Component	Purpose	Example
Lakeflow Connect	Ingest data from external sources	MySQL, SaaS applications
Lakeflow Declarative Pipelines	Build and maintain data transformations	Bronze → Silver → Gold
Lakeflow Jobs	Orchestrate and schedule tasks	Run ingestion, then transformations, then audit

The older name Delta Live Tables (DLT) is now referred to as Lakeflow Declarative Pipelines. You may still see dlt in older projects and interview questions. The newer Python API uses from pyspark import pipelines as dp. 
Databricks on AWS+1

2. Imperative vs declarative pipelines
Imperative approach: You write the steps and control the execution order.
df = spark.read.csv(source_path, header=True)clean_df = df.dropDuplicates()clean_df.write.format("delta").mode("overwrite").saveAsTable(target_table)



You explicitly tell Spark what to do: read, clean, write.

Declarative approach: You define the datasets and their transformations. The pipeline framework determines how to build and update them.
from pyspark import pipelines as dp@dp.materialized_viewdef clean_customers():    return (        spark.read.table("workspace.ecommerce_bronze.customers")        .dropDuplicates(["customer_id"])    )



You describe the desired dataset. Lakeflow manages the pipeline execution and dependencies.
Imperative	Declarative
Explicit read-transform-write steps	Define datasets and transformations
You manage execution logic	Framework manages execution
Often uses notebooks or Python scripts	Uses pipeline definitions
Good for custom procedural logic	Good for managed data pipelines

Interview answer: Declarative pipelines reduce the amount of orchestration and maintenance code you write. They do not eliminate the need to understand transformations, dependencies, data quality or failures. 
Databricks+1

3. Python decorators in Lakeflow
A decorator is a Python feature that adds behavior or metadata to a function. In Lakeflow, decorators identify the type of dataset a function defines.
from pyspark import pipelines as dp



Important decorators
Decorator	Purpose	Typical use
@dp.table	Defines a streaming table	Incremental ingestion
@dp.materialized_view	Defines a materialized view	Batch transformations and aggregates
@dp.temporary_view	Defines a pipeline-scoped temporary view	Intermediate transformations

One correction to your notes: @dp.view is not the current Python API name shown in the current reference. The newer equivalent for a pipeline-scoped view is @dp.temporary_view. Older DLT code commonly used @dlt.view. 
Databricks on AWS+1

Streaming table example — Bronze
from pyspark import pipelines as dp@dp.table(    name="bronze_customers")def bronze_customers():    return (        spark.readStream        .format("cloudFiles")        .option("cloudFiles.format", "csv")        .option("header", "true")        .load("/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/customers/")    )



This defines an incrementally processed streaming table. The pipeline manages its execution and state.
Materialized view example — Gold
from pyspark import pipelines as dp@dp.materialized_view(    name="gold_daily_sales")def gold_daily_sales():    return spark.sql("""        SELECT            order_date,            COUNT(DISTINCT order_id) AS total_orders,            SUM(total_amount) AS revenue        FROM workspace.ecommerce_silver.orders        GROUP BY order_date    """)



This defines a materialized result from a batch query. It is useful for analytical tables and aggregations.
Temporary view example — intermediate transformation
from pyspark import pipelines as dp@dp.temporary_viewdef valid_orders():    return (        spark.read.table("workspace.ecommerce_silver.orders")        .filter("order_status = 'COMPLETED'")    )



This view is an intermediate dataset inside the pipeline, not a persisted Gold table.
4. Streaming table vs materialized view vs temporary view
Feature	Streaming table	Materialized view	Temporary view
Main purpose	Incremental ingestion	Persist query results	Intermediate logic
Data stored as managed target	Yes	Yes	No persistent target
Typical input	Streaming source	Batch query	Batch or pipeline dataset
Example layer	Bronze	Silver or Gold	Between transformations
Typical use	New incoming files	Cleaned or aggregated data	Reusable intermediate query

Important: Don't claim a streaming table processes every record exactly once under every possible failure scenario. Source guarantees, checkpointing, sink behavior and your pipeline design determine the effective processing semantics.
5. Data quality expectations
Expectations are rules that validate records as they move through a pipeline.

For example, in your E-Commerce project:
- customer_id must not be null.
- order_id must not be null.
- total_amount must be greater than zero.
- payment_status must be valid.
Three main expectation behaviors
Expectation	Invalid row behavior	Use case
expect	Keep the row and record quality metrics	Monitor suspicious data
expect_or_drop	Drop the invalid row	Remove invalid records
expect_or_fail	Fail the affected flow update	Critical data must be valid

Example: Silver orders table
from pyspark import pipelines as dp@dp.table(    name="silver_orders")@dp.expect(    "valid_order_id",    "order_id IS NOT NULL")@dp.expect_or_drop(    "valid_customer_id",    "customer_id IS NOT NULL")@dp.expect_or_fail(    "positive_order_amount",    "total_amount > 0")def silver_orders():    return (        spark.readStream        .table("workspace.ecommerce_bronze.orders")        .dropDuplicates(["order_id"])    )



Interpretation:
1. A missing order_id is retained, but the failed expectation is recorded.
2. A missing customer_id causes that row to be dropped.
3. A zero or negative amount causes the affected flow update to fail.
Notice that expect does not clean the data. If invalid rows must be removed, use expect_or_drop or route them to a quarantine design.
expect_all variants
Use these when you have several validation rules.
from pyspark import pipelines as dp@dp.table(name="silver_customers")@dp.expect_all({    "valid_customer_id": "customer_id IS NOT NULL",    "valid_email": "email IS NOT NULL",    "valid_age": "age >= 18"})def silver_customers():    return spark.readStream.table(        "workspace.ecommerce_bronze.customers"    )



The three multiple-rule versions are:
- expect_all: keep invalid rows and log metrics.
- expect_all_or_drop: drop rows that violate expectations.
- expect_all_or_fail: fail the affected flow update when a row violates an expectation.
These rules are evaluated against each record. 
Databricks on AWS+1

6. Lakeflow Jobs — orchestration
A Job is a collection of tasks that Databricks executes according to dependencies and conditions.

Example:
E-Commerce daily workflow
Task 1 — Check landing data

Validate whether today's source files are available


Task 2 — Ingest Bronze

Load customers, products, orders and other datasets


Task 3 — Lakeflow Declarative Pipeline

Build Silver and Gold tables with expectations


Task 4 — Refresh dashboard / success audit


If no data is available → Audit-log task

Record the missing-data event instead of running downstream processing

Conceptual DAG. In the actual job, the conditional branch is configured from the data-availability check.
6.1 Main building blocks
Building block	Meaning	E-Commerce example
Job	Entire workflow	Daily E-Commerce ETL
Task	One unit of work	Bronze ingestion
Dependency	Execution relationship	Silver waits for Bronze
DAG	Directed Acyclic Graph	Full task dependency graph
Trigger	Starts the job	Daily at 1 AM
Parameter	Runtime input	run_date
Retry	Re-executes a failed task	Retry transient ingestion error
Timeout	Maximum allowed duration	Stop a task after its limit
Monitoring	Tracks execution and failures	Job run history and task logs

A DAG is a graph of tasks and dependencies. "Directed" means dependencies have a direction; "acyclic" means there are no circular dependencies. Independent tasks can run in parallel. 
Microsoft Learn+1

6.2 Common task types
Task type	Example
Notebook	Run a PySpark notebook
Python script	Run a .py file
Python wheel	Run a packaged Python application
Pipeline	Run a Lakeflow Declarative Pipeline
SQL	Execute SQL statements or queries
Lakeflow Connect / ingestion pipeline	Ingest from supported external sources
dbt	Execute dbt transformations
Run Job	Trigger another Databricks job
If/else	Branch based on a condition
For each	Repeat a task over multiple inputs
Webhook	Send an HTTP request to an external service
JAR / Spark Submit	Run packaged Spark applications

The exact task options exposed in your workspace can vary by product availability and preview status. 
Databricks on AWS+1

6.3 Job-level settings
Know these for interviews:
- Maximum concurrent runs: Limits how many instances of the same job can run simultaneously.
- Timeout: Prevents a task or job from running indefinitely.
- Retry policy: Retries tasks after failures.
- Notifications: Sends alerts for success, failure or other configured conditions.
- Tags: Categorizes jobs, for example project=ecommerce.
- Permissions: Controls who can view, run, manage or own a job.
- Git: Connects task source code to version control.
- Queueing: Queues runs when capacity or concurrency limits prevent immediate execution.
A key default: new jobs generally allow one concurrent active run. Also, retries are not automatically enabled for every job configuration. Set them deliberately for tasks where retrying is safe. 
Databricks on AWS+1

7. Job triggers and scheduling
Trigger	When it runs	Example
Time-based	At a configured time or interval	Every day at 1 AM
Continuous	Repeatedly keeps the job running	Continuously process incoming events
File arrival	When new files arrive in a monitored Unity Catalog location	New orders CSV arrives
Table update	When monitored source tables change	Bronze orders table updates
Manual	When a user starts it	Test a workflow
Upstream job completion	When another job completes with a matching result	Run reporting after ingestion

Practical choice for your project:
- Use a daily schedule if all datasets arrive at a predictable time.
- Use file-arrival triggers if files arrive irregularly.
- Use table-update triggers if a downstream workflow should start after a source table changes.
- Use continuous execution only when low-latency processing justifies the cost and operational complexity.
File-arrival triggers monitor Unity Catalog volumes or external locations. Table-update triggers can monitor supported Unity Catalog tables and views. 
Databricks on AWS+2

8. Parameters in Jobs and pipelines
Parameters let you reuse the same notebook or pipeline with different inputs.
8.1 Notebook parameters
Create a notebook with this code:
dbutils.widgets.text("run_date", "2026-11-16")dbutils.widgets.text("source_dataset", "orders")run_date = dbutils.widgets.get("run_date")source_dataset = dbutils.widgets.get("source_dataset")print("Run Date:", run_date)print("Dataset:", source_dataset)



When configuring a Notebook task in Lakeflow Jobs, pass:
Parameter	Value
run_date	2026-11-16
source_dataset	orders

The notebook reads those values at runtime.
8.2 Pipeline parameters using spark.conf
For Python-based Lakeflow pipelines, use the pipeline's Configuration settings to expose values through Spark configuration.

Example pipeline configuration:
ecommerce.source_catalog = workspace
ecommerce.source_schema = ecommerce_bronze


Python pipeline code:
from pyspark import pipelines as dp@dp.materialized_viewdef customers_for_analysis():    catalog = spark.conf.get(        "ecommerce.source_catalog"    )    schema = spark.conf.get(        "ecommerce.source_schema"    )    source_table = f"{catalog}.{schema}.customers"    return spark.read.table(source_table)



The Configuration field is the Python approach for this style of parameterization. Databricks also has a separate Pipeline Parameters feature for SQL source code; don't confuse those two mechanisms. 
Databricks on AWS+1

9. Build four reference Lakeflow Jobs
These four designs cover the common interview scenarios. They are separate job examples, not four jobs you must create to build one production workflow.
Job 1 — Notebook job with parameters
Beginner
Purpose: Run a reusable notebook with different input values.

Example task graph:
Job: ECommerce_Notebook_Job

       Start
         |
         v
   Bronze_Ingestion
         |
         v
   Validate_Bronze
         |
         v
      Complete


Build it in the Databricks UI:
1. Open Jobs & Pipelines.
2. Click Create job.
3. Name it ECommerce_Notebook_Job.
4. Add a Notebook task named Bronze_Ingestion.
5. Select your Bronze ingestion notebook.
6. Add notebook parameters such as run_date and source_dataset.
7. Add a second Notebook task named Validate_Bronze.
8. Set Validate_Bronze to depend on Bronze_Ingestion.
9. Add a daily schedule if required.
10. Save and click Run now to test it.
Interview scenario: How do you run the same ingestion notebook for different datasets?

Answer: I pass parameters such as dataset name and run date through the notebook task. The same notebook can process customers, products or orders without duplicating code.
Job 2 — Declarative Pipeline job
Beginner–Intermediate
Purpose: Run a Lakeflow Declarative Pipeline from a Job.

Example:
Job: ECommerce_DLT_Pipeline_Job

          Start
            |
            v
     Lakeflow Pipeline
       /          \
      v            v
  Silver Tables  Gold Tables
      \            /
       v          v
        Pipeline Complete


Build it:
1. Create your Lakeflow Declarative Pipeline.
2. Configure its source notebooks or Python files.
3. Set its target catalog and schema.
4. Add your Bronze-to-Silver and Silver-to-Gold dataset definitions.
5. Add data quality expectations.
6. Save and validate the pipeline.
7. Open Jobs & Pipelines and create a new Job.
8. Add a Pipeline task and select your existing pipeline.
9. Configure the pipeline update mode and any supported parameters.
10. Save, run and inspect the result.
Interview scenario: Why use a pipeline task instead of calling a notebook manually?

Answer: A pipeline task executes the managed declarative pipeline. It provides pipeline-level execution, dependency management, data quality metrics and pipeline monitoring rather than requiring me to manually sequence every transformation notebook.
Job 3 — Ingestion + ETL job
Intermediate
Purpose: Coordinate ingestion and transformation in one DAG.
Job: ECommerce_End_To_End_ETL

            Start
              |
              v
        Ingest_Customers
              |
              v
         Ingest_Orders
              |
              v
        Ingest_Payments
              |
              v
        Lakeflow_Pipeline
              |
              v
         Refresh_Gold
              |
              v
         Success_Audit


You can also run independent ingestion tasks in parallel:
                  Start
                    |
          +---------+---------+
          |         |         |
          v         v         v
      Customers   Products   Orders
          |         |         |
          +---------+---------+
                    |
                    v
             Lakeflow Pipeline
                    |
                    v
               Gold Tables


Build it:
1. Create the three ingestion tasks.
2. Set them to run in parallel if their source data is independent.
3. Add the Lakeflow Pipeline task.
4. Make the pipeline depend on all required ingestion tasks.
5. Add a Gold validation or dashboard refresh task.
6. Add a success audit task.
7. Configure retries and task timeouts.
8. Add a daily schedule or event-based trigger.
Important: Do not run dependent tasks in parallel just to make the job faster. Silver transformations must wait until their required Bronze inputs are ready.
Job 4 — Conditional load and audit-log job
Intermediate–Advanced
Purpose: Check whether data is available. If it is, process it. If not, record the reason and stop downstream processing.
              Start
                |
                v
         Check_Data_Available
                |
                v
             If / Else
             /       \
          YES         NO
           |           |
           v           v
     Ingest_Bronze   Audit_Log
           |
           v
    Lakeflow_Pipeline
           |
           v
       Success_Audit


Step 1 — Create the data-availability notebook

For example, use a notebook task that checks whether the expected source files exist in your landing folders.
datasets = [    "customers",    "products",    "orders",    "order_items",    "payments",    "shipments",    "inventory_snapshots"]landing_path = (    "/Volumes/workspace/default/data_volume/"    "ecommerce_lakehouse/landing")available_datasets = []missing_datasets = []for dataset in datasets:    file_path = (        f"{landing_path}/{dataset}/{dataset}.csv"    )    try:        dbutils.fs.ls(file_path)        available_datasets.append(dataset)    except Exception:        missing_datasets.append(dataset)data_available = len(missing_datasets) == 0print("Available:", available_datasets)



This example checks all seven expected files. It is a simple file-existence check, not a complete validation of file contents, schema or completeness.

Step 2 — Configure the If/else task

In the Job graph:
1. Add an If/else task named Data_Available_Check.
2. Set its dependency to Check_Data_Available.
3. Configure its condition to read the data_available task value.
4. Configure the true branch to run Ingest_Bronze.
5. Configure the false branch to run Audit_Log.
Step 3 — Create the audit-log notebook
from datetime import datetimemissing_datasets = dbutils.jobs.taskValues.get(    taskKey="Check_Data_Available",    key="missing_datasets")audit_record = [(    datetime.now().isoformat(),    "DATA_NOT_AVAILABLE",    missing_datasets)]audit_df = spark.createDataFrame(    audit_record,    [        "event_timestamp",        "status",        "missing_datasets"    ])audit_df.write \    .format("delta") \    .mode("append") \    .saveAsTable(        "workspace.ecommerce_silver.audit_log"    )print("Missing-data event recorded.")



Create the audit table first, with compatible column types, if it doesn't already exist.

Step 4 — Add the remaining dependencies
- Ingest_Bronze runs only on the true branch.
- Lakeflow_Pipeline depends on successful Bronze ingestion.
- Success_Audit depends on successful pipeline completion.
- Audit_Log runs on the false branch.
Interview scenario: What happens if the orders file is missing?

Answer: The availability check identifies the missing file and sets a task value. The If/else task reads that value. The false branch writes an audit record, and the ingestion and transformation tasks are skipped. This avoids processing an incomplete batch.
10. Build and monitor a high-complexity multi-task workflow
This is the most important practical exercise for your E-Commerce Lakehouse interview.
Recommended hands-on exercise
10.1 Target workflow
Build one job called:

ECommerce_Lakehouse_High_Multi_Task_Workflow

Use this task design:
Task key	Task type	Dependency
check_data	Notebook	None
data_condition	If/else	check_data
ingest_customers	Notebook	True branch
ingest_products	Notebook	True branch
ingest_orders	Notebook	True branch
ingest_payments	Notebook	True branch
run_pipeline	Pipeline	All required ingestion tasks
validate_gold	SQL or Notebook	run_pipeline
success_audit	Notebook	validate_gold
missing_data_audit	Notebook	False branch

You can add shipments, order items and inventory ingestion tasks using the same pattern.
10.2 Create the workflow in the UI
1. Create the job
   
   Open Databricks → Jobs & Pipelines → Create job. Give it the name ECommerce_Lakehouse_High_Multi_Task_Workflow.
2. Add the data check
   
   Add a Notebook task and select your data-availability notebook. Set the task key to check_data.
3. Add the conditional task
   
   Add an If/else task. Make it depend on check_data and configure its condition using the data_available task value.
4. Add ingestion tasks
   
   Add the customer, product, order and payment ingestion notebooks to the true branch. Run independent ingestion tasks in parallel.
5. Add the pipeline task
   
   Add a Pipeline task that runs your Lakeflow Declarative Pipeline. Make it depend on all the ingestion tasks that it requires.
6. Add validation and audit
   
   Add a SQL or Notebook task to validate Gold tables. Add a success-audit notebook after validation. Configure the false branch to run the missing-data audit notebook.
7. Configure reliability settings
   
   Set retries for suitable transient failures, task timeouts, job notifications, permissions and a maximum concurrent run limit.
8. Configure the trigger
   
   For an initial test, use manual execution. After validation, add a schedule or a file-arrival trigger appropriate to your data arrival pattern.
10.3 Monitor the job
After clicking Run now, inspect these areas:
Monitoring area	What to check
Graph view	Which tasks ran, succeeded, failed or were skipped
Run details	Start time, end time and total duration
Task output	Printed messages and notebook output
Task logs	Exceptions and execution errors
Pipeline details	Table updates, data quality metrics and failed flows
Event log	Pipeline events and execution details
Notifications	Failure and success alerts

10.4 What to do when a task fails
Example: ingest_orders fails because of a temporary source-storage issue.
1. Open the failed job run.
2. Select ingest_orders.
3. Inspect the error message and logs.
4. Fix the underlying issue.
5. Use the repair-run functionality to rerun the failed task and the required downstream tasks, rather than blindly rerunning the entire workflow.
6. Confirm that the pipeline and Gold tables are correct.
Interview point: Retries help with temporary failures, but they do not fix bad data or broken application logic. Design ingestion and writes to be idempotent so retrying a task does not create duplicate records. 
Databricks on AWS+1

11. DAG view vs lineage view
These two concepts are related, but they answer different questions.
DAG view	Data lineage view
Shows task execution dependencies	Shows data movement and dependencies
Belongs to Jobs	Available through Unity Catalog and pipeline lineage
Answers: What runs first?	Answers: Where did this table's data come from?
Example: Bronze ingestion before pipeline	Example: Orders CSV → Bronze orders → Silver orders → Gold daily sales

Example:
Job DAG
Ingest Bronze

Run Pipeline

Validate Gold

Data lineage
Bronze Orders

Silver Orders

Gold Daily Sales

12. Interview questions and answers
Q1. What is Lakeflow Jobs?

Lakeflow Jobs is Databricks' orchestration service. It schedules and executes tasks such as notebooks, SQL queries, Python scripts and pipelines. It manages dependencies, retries, conditional execution and monitoring.

Q2. What is a DAG?

A DAG is a Directed Acyclic Graph that represents tasks and their dependencies. It determines execution order and allows independent tasks to run in parallel.

Q3. What is the difference between a Job and a Task?

A Job is the complete workflow. A Task is an individual operation inside that workflow.

Q4. What happens when an upstream task fails?

By default, dependent tasks that require successful completion do not proceed. Depending on the configured dependency and run-if condition, other tasks such as cleanup or audit logging can still run.

Q5. What is a retry policy?

A retry policy automatically reruns a failed task a configured number of times, often with an interval between attempts. It is useful for temporary infrastructure or network failures.

Q6. What is the difference between a schedule and a file-arrival trigger?

A schedule starts a job at a defined time. A file-arrival trigger starts a job when new files arrive in a monitored Unity Catalog storage location.

Q7. What is Lakeflow Declarative Pipelines?

It is Databricks' managed declarative data-pipeline framework. We define streaming tables, materialized views and data quality expectations, while the framework manages pipeline execution.

Q8. What is the difference between @dp.table and @dp.materialized_view?

@dp.table defines a streaming table, commonly used for incremental ingestion. @dp.materialized_view defines a materialized dataset from a batch query, commonly used for transformations and aggregations.

Q9. What is an expectation?

An expectation is a data quality constraint. It validates records and can keep invalid records while logging metrics, drop them, or fail the affected flow update.

Q10. How do you handle missing input files in a workflow?

I use a data-availability check notebook and an If/else task. If all required files are available, ingestion and transformations proceed. Otherwise, an audit task records the missing files and downstream processing is skipped.

Q11. What is the difference between expect_or_drop and expect_or_fail?

expect_or_drop removes invalid rows. expect_or_fail fails the affected flow update when a row violates the expectation.

Q12. How do you monitor a Lakeflow Job?

I inspect its graph, run history, task status, task output, logs, execution duration, notifications and pipeline event logs or data quality metrics.

Q13. What is idempotency in orchestration?

Idempotency means that rerunning a task with the same input does not create unintended duplicate or inconsistent results. It is important because tasks can be retried or repaired after failure.

Q14. Can two tasks run simultaneously?

Yes. Independent tasks can run in parallel. Tasks with dependencies wait for their upstream requirements.

Q15. How do you explain your E-Commerce orchestration design in an interview?

I use Lakeflow Jobs to orchestrate ingestion, transformations and validation. Independent datasets are ingested in parallel. Once all required Bronze ingestion tasks finish successfully, the Lakeflow Declarative Pipeline processes Silver and Gold. I use expectations for data quality, conditional tasks for missing data, retries for transient failures and audit tables for execution tracking.